# Seed media — 61 photographs for brgen, amber and bsdports

**Runtime → Change runtime type → T4 GPU** before running.

Set `HF_TOKEN` in the sidebar (🔑) to a token that has accepted the
[FLUX.1-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-dev).
A token without it downloads a 403 that surfaces hundreds of lines later
inside diffusers, so the setup cell checks for it before anything else.

Three passes, in this order, because the adapter is loaded once and unloading
it costs a reload:

1. **dating** — 12 frames, Ragnhild's adapter at weight
   0.85
2. **scenes** — 32 frames, base model, no adapter
3. **amber** — 17 frames, base model, the one mannequin

Roughly 20–40 s a frame on a T4 at nf4, so about half an hour of GPU plus the
model download. Output goes to Drive; a disconnect costs the frames since the
last write and not the session.

Prompts are `STUDIO/lora/seed_media.yml`. Edits belong there — this notebook is
generated by `_toolkit/run_seed_media_colab.rb` and is overwritten.


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("ok: HF_TOKEN from Colab secrets")
except Exception:
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (FLUX.1-dev licence accepted): ")
assert os.environ.get("HF_TOKEN"), "no HF token"

# Fail here, not four hundred lines into a download. A token that has not
# accepted the licence gets 403 on the weights and the traceback names a
# missing file rather than a missing acceptance.
import urllib.request
req = urllib.request.Request(
    "https://huggingface.co/api/models/black-forest-labs/FLUX.1-dev",
    headers={"Authorization": f"Bearer {os.environ['HF_TOKEN']}"})
try:
    urllib.request.urlopen(req).read(1)
    print("ok: FLUX.1-dev reachable with this token")
except Exception as e:
    raise SystemExit(
        "FLUX.1-dev is not reachable with this token. Accept the licence at "
        "https://huggingface.co/black-forest-labs/FLUX.1-dev and rerun. " + str(e))


In [ ]:
# Frames go to Drive as they are made. A free session disconnects when idle and
# is capped near 12 h; writing at the end would make every disconnect cost the
# whole run.
import os
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/seed_media"
os.makedirs(OUT, exist_ok=True)
print("ok:", OUT)


In [ ]:
import subprocess, os
# bitsandbytes for nf4, peft for the adapter, sentencepiece for T5's tokenizer.
subprocess.run(
    "pip -q install -U diffusers transformers accelerate safetensors "
    "bitsandbytes peft sentencepiece protobuf",
    shell=True, check=True)

if not os.path.isdir("/content/pub4/.git"):
    subprocess.run(
        "git clone --depth 1 --branch main https://github.com/anon987654321/pub4.git /content/pub4",
        shell=True, check=True)
print("ok: toolkit and prompts at /content/pub4")


In [ ]:
import gc, os, yaml, torch
from diffusers import FluxPipeline, BitsAndBytesConfig, FluxTransformer2DModel
from transformers import T5EncoderModel, BitsAndBytesConfig as TfBitsAndBytesConfig

SPEC = "/content/pub4/STUDIO/lora/seed_media.yml"
spec = yaml.safe_load(open(SPEC))
MODEL = "black-forest-labs/FLUX.1-dev"

# float16, not bfloat16. A T4 is Turing (sm_75) and has no bf16 at all, which is
# the single fact that decides every dtype below.
DTYPE = torch.float16

# Both the transformer and T5 are quantised, and both have to be. nf4 puts the
# 12B transformer near 6.5 GB, but T5-XXL is another 9 GB in fp16 and 16 GB does
# not hold both plus activations. Quantised together they fit with room for the
# VAE decode.
nf4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE)
tf_nf4 = TfBitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=DTYPE)

transformer = FluxTransformer2DModel.from_pretrained(
    MODEL, subfolder="transformer", quantization_config=nf4, torch_dtype=DTYPE)
text_encoder_2 = T5EncoderModel.from_pretrained(
    MODEL, subfolder="text_encoder_2", quantization_config=tf_nf4, torch_dtype=DTYPE)

pipe = FluxPipeline.from_pretrained(
    MODEL, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=DTYPE)
# Offload rather than .to("cuda"): it keeps only the active submodule resident,
# which is what leaves headroom for the VAE at 1024px.
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)
print("ok: FLUX.1-dev loaded nf4")

RATIOS = {"3:2": (1216, 832), "4:5": (896, 1120), "1:1": (1024, 1024), "4:3": (1152, 864)}

def size_for(ratio):
    return RATIOS.get(ratio or spec["meta"]["aspect_ratio"], (1024, 1024))

def render(key, prompt, ratio, seed):
    path = os.path.join(OUT, key + ".png")
    # Resume is free and a disconnect is not. Re-running skips what landed.
    if os.path.exists(path):
        print("skip", key); return
    w, h = size_for(ratio)
    image = pipe(prompt=prompt, width=w, height=h,
                 guidance_scale=spec["meta"]["guidance"],
                 num_inference_steps=spec["meta"]["steps"],
                 generator=torch.Generator("cpu").manual_seed(seed)).images[0]
    image.save(path)
    print("ok", key, f"{w}x{h}")

# Pass 1 — dating. The adapter is loaded once for the whole pass and unloaded
# after, because two thirds of this set must render without it.
dating = spec["dating"]
ADAPTER = "/content/pub4/STUDIO/lora/ragnhild/weights/ragnhild/ragnhild.safetensors"
if os.path.exists(ADAPTER):
    # set_adapters, never fuse_lora. Fusing writes the adapter into the base
    # weights, and these are nf4 -- bitsandbytes 4-bit layers cannot be
    # written back into, so fuse raises partway through the dating pass, on
    # the GPU, after the model has already downloaded. Scaling a named
    # adapter leaves the quantised weights untouched.
    pipe.load_lora_weights(ADAPTER, adapter_name="subject")
    pipe.set_adapters(["subject"], adapter_weights=[dating["lora_weight"]])

    # The trigger comes from subject.env, the file the adapter was trained
    # against. Hardcoding it is how a rename yields twelve pictures of
    # nobody in particular.
    env_path = "/content/pub4/STUDIO/lora/" + dating["lora"] + "/subject.env"
    trigger = dating["lora"]
    if os.path.exists(env_path):
        for line in open(env_path):
            if line.startswith("TRIGGER="):
                trigger = line.split("=", 1)[1].strip().strip('"').strip("'")
    print("ok: adapter loaded, trigger =", trigger)

    for i, (key, prompt) in enumerate(dating["profiles"].items()):
        render(key, prompt.replace("TRIGGER", trigger), dating["aspect_ratio"], 1000 + i)

    pipe.delete_adapters("subject")
    gc.collect(); torch.cuda.empty_cache()
    print("ok: dating pass complete, adapter unloaded")
else:
    # Named loudly. Rendering these on the base model produces twelve strangers
    # that look like a result and are not one.
    print("SKIPPED dating: no adapter at", ADAPTER)

# Pass 2 — scenes. No adapter, no face where a face is not the subject.
for i, (key, entry) in enumerate(spec["scenes"].items()):
    render(key, entry["prompt"], entry.get("aspect_ratio"), 2000 + i)

# Pass 3 — amber. The mannequin clause is prepended rather than repeated in
# each entry, so seventeen garments share one mannequin instead of seventeen
# subtly different ones.
amber = spec["amber"]
for i, (key, garment) in enumerate(amber["garments"].items()):
    render(key, spec["mannequin"] + ", " + garment, amber["aspect_ratio"], 3000 + i)

made = len([f for f in os.listdir(OUT) if f.endswith(".png")])
print(f"\nok: {made} frame(s) in {OUT}")
print("next: download that folder, then on the Mac:")
print("  ruby STUDIO/lora/_toolkit/install_seed_media.rb <folder>")
